# Homework: Vector Search

LLM Zoomcamp 2026 — Module 2.

We turn text into vectors with a lightweight ONNX embedder, do vector search
by hand and with `minsearch`, and compare it with keyword search.

## Setup

Dependencies (installed via `uv add onnxruntime tokenizers numpy tqdm minsearch gitsource`):
- `embedder.py` / `download.py` from the course repo (`02-vector-search/embed/`)
- ONNX model `Xenova/all-MiniLM-L6-v2` (already downloaded with `python download.py`)

In [1]:
import numpy as np

from embedder import Embedder
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index, VectorSearch

emb = Embedder()

---
## Q1. Embedding a query

Embed the query:

> How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value (`v[0]`)?

- -0.31
- -0.02
- 0.12
- 0.44

In [2]:
v = emb.encode("How does approximate nearest neighbor search work?")
print("vector shape:", v.shape)
print("v[0]:", round(float(v[0]), 4))

vector shape: (384,)
v[0]: -0.0206


**Answer: -0.02** (v[0] ≈ -0.0206)

## Loading the data

Pull the 72 lesson pages from the course repo at commit `8c1834d`.

In [3]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(f"Loaded {len(documents)} documents")

Loaded 72 documents


---
## Q2. Cosine similarity

The embedder returns normalized vectors, so the dot product is the cosine
similarity.

Take the page `02-vector-search/lessons/07-sqlitesearch-vector.md`, embed
its `content`, and compute the cosine similarity with the query vector from
Q1.

- 0.07
- 0.37
- 0.68
- 0.92

In [4]:
target = "02-vector-search/lessons/07-sqlitesearch-vector.md"
page = next(d for d in documents if d["filename"] == target)

page_vector = emb.encode(page["content"])
cosine = float(v.dot(page_vector))
print("cosine similarity:", round(cosine, 4))

cosine similarity: 0.3611


**Answer: 0.37** (≈ 0.3611)

---
## Q3. Chunking and search by hand

Chunk the pages (`size=2000, step=1000`), embed every chunk's `content`
with `encode_batch`, stack into a matrix `X`, and score the Q1 query
against all chunks: `scores = X.dot(v)`.

Which file does the highest-scoring chunk belong to?

- `02-vector-search/lessons/03-embeddings-dataset.md`
- `02-vector-search/lessons/06-rag-vector.md`
- `02-vector-search/lessons/07-sqlitesearch-vector.md`
- `02-vector-search/lessons/09-onnx-embedder.md`

In [5]:
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"{len(chunks)} chunks")

texts = [c["content"] for c in chunks]
X = emb.encode_batch(texts)
print("X shape:", X.shape)

scores = X.dot(v)
best = int(np.argmax(scores))
print("best score:", round(float(scores[best]), 4))
print("best filename:", chunks[best]["filename"])

295 chunks


X shape: (295, 384)
best score: 0.6489
best filename: 02-vector-search/lessons/07-sqlitesearch-vector.md


**Answer: `02-vector-search/lessons/07-sqlitesearch-vector.md`**

---
## Q4. Vector search with minsearch

Use `VectorSearch` from minsearch on the chunk vectors and search:

> What metric do we use to evaluate a search engine?

Which file is the `filename` of the first result?

- `02-vector-search/lessons/04-vector-search.md`
- `04-evaluation/lessons/05-search-metrics.md`
- `04-evaluation/lessons/13-llm-as-judge.md`
- `05-monitoring/lessons/04-metrics.md`

In [6]:
vs = VectorSearch(keyword_fields=["filename"])
vs.fit(X, chunks)

q4_vec = emb.encode("What metric do we use to evaluate a search engine?")
results4 = vs.search(q4_vec, num_results=5)
print("first filename:", results4[0]["filename"])

first filename: 04-evaluation/lessons/05-search-metrics.md


**Answer: `04-evaluation/lessons/05-search-metrics.md`**

---
## Q5. Text search vs vector search

Index the same chunks with `Index` (keyword search, `content` as a text
field). Run both searches for:

> How do I store vectors in PostgreSQL?

Take the top 5 results from each. Which file shows up in the vector
results but not in the text results?

- `02-vector-search/lessons/01-intro.md`
- `02-vector-search/lessons/02-embeddings.md`
- `02-vector-search/lessons/08-pgvector.md`
- `03-orchestration/lessons/05-rag.md`

In [7]:
# Vector search
q5_vec = emb.encode("How do I store vectors in PostgreSQL?")
vec_results = vs.search(q5_vec, num_results=5)

# Text search
text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)
text_results = text_index.search("How do I store vectors in PostgreSQL?", num_results=5)

vec_files = [d["filename"] for d in vec_results]
text_files = [d["filename"] for d in text_results]

print("Vector top-5 files:")
for f in vec_files:
    print("  -", f)
print("\nText top-5 files:")
for f in text_files:
    print("  -", f)

print("\nIn vector but not in text:")
for f in vec_files:
    if f not in text_files:
        print("  -", f)

Vector top-5 files:
  - 02-vector-search/lessons/08-pgvector.md
  - 02-vector-search/lessons/08-pgvector.md
  - 03-orchestration/lessons/05-rag.md
  - 02-vector-search/lessons/08-pgvector.md
  - 02-vector-search/lessons/08-pgvector.md

Text top-5 files:
  - 02-vector-search/lessons/02-embeddings.md
  - 03-orchestration/lessons/05-rag.md
  - 02-vector-search/lessons/01-intro.md
  - 03-orchestration/lessons/05-rag.md
  - 02-vector-search/lessons/01-intro.md

In vector but not in text:
  - 02-vector-search/lessons/08-pgvector.md
  - 02-vector-search/lessons/08-pgvector.md
  - 02-vector-search/lessons/08-pgvector.md
  - 02-vector-search/lessons/08-pgvector.md


**Answer: `02-vector-search/lessons/08-pgvector.md`**

Vector search finds the pgvector page by meaning even though it doesn't
share exact words with the query; keyword search misses it.

---
## Q6. Hybrid search

Vector search matches by meaning, keyword search by exact words. We combine
them with **Reciprocal Rank Fusion (RRF)**, which ignores raw scores and
uses only the position (`rank`, starting at 0) of each document in each list:

```
RRF(d) = sum over lists of  1 / (k + rank(d))    with k = 60
```

Run the query `How do I give the model access to tools?` with vector and
text search, fuse the results, and find the top file.

- `01-agentic-rag/lessons/01-intro.md`
- `01-agentic-rag/lessons/13-function-calling.md`
- `01-agentic-rag/lessons/14-agentic-loop.md`
- `01-agentic-rag/lessons/16-other-frameworks.md`

In [8]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [9]:
q6 = "How do I give the model access to tools?"

q6_vec = emb.encode(q6)
vector_results = vs.search(q6_vec, num_results=5)
text_results = text_index.search(q6, num_results=5)

print("Vector top-5:")
for d in vector_results:
    print(f"  {d['filename']} (start={d['start']})")
print("\nText top-5:")
for d in text_results:
    print(f"  {d['filename']} (start={d['start']})")

fused = rrf([vector_results, text_results])
print("\nFused top-5:")
for d in fused:
    print(f"  {d['filename']} (start={d['start']})")
print("\nFirst after RRF:", fused[0]["filename"])

Vector top-5:
  01-agentic-rag/lessons/01-intro.md (start=2000)
  04-evaluation/lessons/02-ground-truth.md (start=1000)
  01-agentic-rag/lessons/16-other-frameworks.md (start=0)
  01-agentic-rag/lessons/15-frameworks.md (start=2000)
  01-agentic-rag/lessons/13-function-calling.md (start=4000)

Text top-5:
  01-agentic-rag/lessons/14-agentic-loop.md (start=0)
  01-agentic-rag/lessons/13-function-calling.md (start=4000)
  01-agentic-rag/lessons/13-function-calling.md (start=5000)
  01-agentic-rag/lessons/13-function-calling.md (start=1000)
  04-evaluation/lessons/02-ground-truth.md (start=3000)

Fused top-5:
  01-agentic-rag/lessons/13-function-calling.md (start=4000)
  01-agentic-rag/lessons/01-intro.md (start=2000)
  01-agentic-rag/lessons/14-agentic-loop.md (start=0)
  04-evaluation/lessons/02-ground-truth.md (start=1000)
  01-agentic-rag/lessons/16-other-frameworks.md (start=0)

First after RRF: 01-agentic-rag/lessons/13-function-calling.md


**Answer: `01-agentic-rag/lessons/13-function-calling.md`**

It isn't first in either search on its own, but it ranks high in both, so
RRF promotes it to the top.

---
## Summary

| Question | Answer |
|----------|--------|
| Q1. `v[0]` | **-0.02** |
| Q2. Cosine similarity | **0.37** |
| Q3. Top chunk filename | **`02-vector-search/lessons/07-sqlitesearch-vector.md`** |
| Q4. First vector result | **`04-evaluation/lessons/05-search-metrics.md`** |
| Q5. In vector, not text | **`02-vector-search/lessons/08-pgvector.md`** |
| Q6. First after RRF | **`01-agentic-rag/lessons/13-function-calling.md`** |